> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports (active)

```python
from mrta.retrieval import Embedder, VectorStore
from mrta.core.llm import LLMClient
from mrta.core.rag_pipeline import rag_query
from mrta.prompts import load_prompt
from mrta.observability.logging import StructuredLogger
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 04.

# Phase 4 — End-to-End RAG Pipeline (with Ollama)

**Goal:** Glue together everything from Phases 1–3 into a single function that takes a question and a corpus, retrieves the most relevant chunks, and returns a grounded answer with page citations — running entirely on a local Ollama model.

This is the **first real milestone**: the moment the project becomes useful to a human.

```
Question
   |
   v
Embed question  ->  FAISS search  ->  top-k chunks
                                          |
                                          v
                              build prompt (system + context + question)
                                          |
                                          v
                                       Ollama LLM
                                          |
                                          v
                            Answer + parsed [page X] citations
```


## 4.1 — Reload the vector store


In [ ]:
import os, sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from mrta.retrieval import Embedder, VectorStore

embedder = Embedder()
store = VectorStore.load("data/vector_store/aiayn", embedder)
print("Loaded vector store:", store._index.ntotal, "vectors")

## 4.2 — The prompt template

A good RAG prompt has four ingredients:

1. **Role / behavior** — what kind of answers you want.
2. **Grounding rule** — tell the model to refuse if context is insufficient.
3. **Context block** — the retrieved chunks, each labeled with its source.
4. **Citation format** — explicit format the model must use.

We make this a Jinja2 template so it lives next to other prompts in `src/mrta/prompts/`.


In [ ]:
from mrta.prompts import load_prompt

# Demo: render the template with a single placeholder chunk
demo_chunk_dict = {"source": "x.pdf", "page": 1, "text": "demo"}

class _ChunkProxy:
    """Minimal proxy so the Jinja2 template can access .source / .page / .text attributes."""
    def __init__(self, d):
        self.source = d["source"]; self.page = d["page"]; self.text = d["text"]

print(load_prompt("rag", chunks=[_ChunkProxy(demo_chunk_dict)], question="?")[:400])

## 4.3 — The LLM client wrapper

`src/mrta/core/llm.py` exposes a single `chat(messages, **opts)` function so the rest of the code does not care which provider is behind it.


In [ ]:
from mrta.core.llm import LLMClient

llm = LLMClient()  # reads settings.llm_provider and settings.ollama_llm_model
print("LLM client ready:", llm.model)

## 4.4 — The RAG pipeline as one function


In [ ]:
from mrta.core.rag_pipeline import rag_query

## 4.5 — Try it


In [ ]:
out = rag_query("What is multi-head attention and why is it used?", vector_store=store, llm=llm, top_k=5)

print("Q: What is multi-head attention and why is it used?")
print("\nA:", out["answer"])
print("\nSources (pages):", [c.page for c in out["sources"]])
print(f"Latency: {out['latency_s']:.1f}s")

## 4.6 — A few more questions


In [ ]:
questions = [
    "What dataset and tokenization were used for English-to-German training?",
    "How does scaled dot-product attention scale with sequence length?",
    "Why use sinusoidal positional encodings instead of learned ones?",
    "What were the BLEU scores reported on WMT 2014 English-to-French?",
]

for q in questions:
    out = rag_query(q, vector_store=store, llm=llm, top_k=5)
    print(f"Q: {q}")
    print(f"A: {out['answer'][:280]}{'...' if len(out['answer']) > 280 else ''}")
    print(f"   sources={[c.page for c in out['sources']]}  latency={out['latency_s']:.1f}s\n")

## 4.7 — Observability: log every run

Senior-engineer move. Every answer becomes a structured JSON line we can inspect, replay, and feed into evaluation.


In [ ]:
from mrta.observability.logging import StructuredLogger

logger = StructuredLogger()

# Re-run the first question and log it.
out = rag_query("What is multi-head attention and why is it used?", vector_store=store, llm=llm)
logger.log_run(
    question="What is multi-head attention and why is it used?",
    answer=out["answer"],
    sources=out["sources"],
    latency_s=out["latency_s"],
)
from mrta.core.config import settings
print(f"Logged 1 run → {settings.log_file}")

## 4.8 — Failure modes to watch for

| Symptom                              | Likely cause                                                 | Fix                                                  |
|--------------------------------------|--------------------------------------------------------------|------------------------------------------------------|
| "I could not find this..." too often | k too low, or chunk size too small                           | Raise `k`, or rerun chunking with `chunk_size=900`.  |
| Confident but wrong answers          | LLM ignoring context                                         | Strengthen system prompt; lower temperature; add a re-ask "is this in the context?" pass. |
| Cites the wrong page                 | Citation parser sees a page number inside the answer text    | Make the model emit citations only as `[page N]`, and post-validate against retrieved pages. |
| Cuts mid-sentence                    | LLM context too small for chunks + question + answer         | Reduce `k`, or summarize chunks before feeding.       |

We will quantify all of these in Notebook 09 with Ragas.


## What you learned

- The standard RAG flow: embed → retrieve → prompt → answer + parse citations.
- A Jinja2 prompt template you can iterate on.
- An LLM client wrapper that returns latency + token counts.
- Why structured JSONL logging is a senior-engineer move.
- Common RAG failure modes and what to tune.

## Exercises

1. Add a `reranker` step: re-score the top-20 chunks with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) and keep the top-5. Measure latency vs quality.
2. Add a "no context" guard: if the highest retrieval score is below a threshold (say 0.3), short-circuit and answer "I am not confident I can answer from these documents."
3. Make the citation parser also validate that every cited page was actually retrieved.

## Next: [Phase 5 — FastAPI backend](./2026-05-25-phase05-fastapi-backend.ipynb)
